In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# install dependencies
!pip install -q -U bitsandbytes peft accelerate transformers datasets trl

In [3]:
# imports and Google Drive mount (for checkpointing across sessions)
import json
import torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = Path("/content/drive/MyDrive/finetuning_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Checkpoints will save to: {CHECKPOINT_DIR}")

Mounted at /content/drive
Checkpoints will save to: /content/drive/MyDrive/finetuning_checkpoints


In [4]:
# authenticate with Hugging Face (needed for gated models like Llama)
from huggingface_hub import login
login()  # opens an interactive prompt to paste your HF token

In [5]:
# upload training_examples.json (from the local notebook's output)
from google.colab import files
uploaded = files.upload()  # select finetuning/training_examples.json

with open("training_examples.json", encoding="utf-8") as f:
    training_examples = json.load(f)

print(f"{len(training_examples)} training examples loaded")

Saving training_examples.json to training_examples (1).json
43 training examples loaded


In [6]:
# convert to instruction-tuning format
# Each example becomes a (system, user, assistant) chat triple matching
# your existing CITATION_SYSTEM_PROMPT structure, so the fine-tuned model
# learns to answer in the same format your pipeline already expects

CITATION_SYSTEM_PROMPT = """You are a question-answering assistant that MUST
ground every claim in the provided context chunks.

Rules:
- Answer using ONLY information in the context chunks below.
- Cite every claim inline using the chunk's bracketed number, e.g. [1], [2].
- If multiple chunks support a claim, cite all of them, e.g. [1][3].
- Do NOT state anything not directly supported by the context — do not invent
  facts, names, or events absent from the chunks."""

def format_example(ex):
    context_str = "\n\n".join(
        f"[{i+1}] (source: {c['source_doc']})\n{c['text']}"
        for i, c in enumerate(ex["context_chunks"])
    )
    user_content = f"Context:\n{context_str}\n\nQuestion: {ex['query']}"
    return {
        "messages": [
            {"role": "system", "content": CITATION_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": ex["answer"]},
        ]
    }

formatted = [format_example(ex) for ex in training_examples]
print(f"Formatted {len(formatted)} examples")
print("\nExample 0 preview:")
print(formatted[0]["messages"][-1]["content"][:200])

Formatted 43 examples

Example 0 preview:
Sydney Carton's friend, Mr. Stryver, said that he is "a man already pretty well off, and a rapidly rising man, and a man of some distinction" [1], implying that his prospects are good and that he has 


In [7]:
# train/val split (small dataset, so keep val minimal but non-zero —
# enough to sanity-check loss isn't diverging, not enough to be a real eval)
import random
random.seed(42)
random.shuffle(formatted)

n_val = max(3, len(formatted) // 10)  # ~10%, minimum 3
val_data = formatted[:n_val]
train_data = formatted[n_val:]

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

Train: 39 | Val: 4


In [8]:
# load base model in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Base model loaded in 4-bit")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base model loaded in 4-bit


In [9]:
# attach LoRA adapters
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [10]:
# tokenize the dataset
from datasets import Dataset

MAX_SEQ_LEN = 1536  # per earlier sizing discussion — chunk context eats length fast

def tokenize_example(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    tokenized = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN)  # no padding here
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = Dataset.from_list(train_data).map(tokenize_example, remove_columns=["messages"])
val_dataset = Dataset.from_list(val_data).map(tokenize_example, remove_columns=["messages"])

print(f"Tokenized — train: {len(train_dataset)}, val: {len(val_dataset)}")

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenized — train: 39, val: 4


In [16]:
# training arguments, sized for a 43-example dataset on a T4
# With this few examples, more epochs at a small effective batch is more
# appropriate than fewer epochs at a large batch — LoRA needs repeated
# exposure to shift behavior when the dataset itself is small
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers import DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR / "run1"),
    num_train_epochs=12,   # was 6 — val loss was still dropping, give it more room
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# train. Checkpoints save to Drive every epoch via save_strategy above,
# so if Colab disconnects mid-run, resume=True below picks up from the last
# saved epoch rather than starting over.
import os

last_checkpoint = None
run_dir = CHECKPOINT_DIR / "run1"
if run_dir.exists():
    checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    if checkpoints:
        last_checkpoint = str(checkpoints[-1])
        print(f"Resuming from {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)

Resuming from /content/drive/MyDrive/finetuning_checkpoints/run1/checkpoint-30


Epoch,Training Loss,Validation Loss
7,1.722376,1.805523
8,1.487603,1.796676
9,1.296128,1.794383
10,1.389044,1.793015
11,1.394706,1.795018


In [4]:
# find and confirm the epoch-10 checkpoint
run_dir = CHECKPOINT_DIR / "run1"
checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
for cp in checkpoints:
    trainer_state = cp / "trainer_state.json"
    if trainer_state.exists():
        import json
        state = json.loads(trainer_state.read_text())
        print(cp.name, "-> epoch", state.get("epoch"))

checkpoint-45 -> epoch 9.0
checkpoint-50 -> epoch 10.0
checkpoint-55 -> epoch 11.0


In [8]:
BEST_CHECKPOINT = run_dir / "checkpoint-50"  # epoch 10, lowest val loss

FINAL_ADAPTER_PATH = CHECKPOINT_DIR / "final_adapter_epoch10"
import shutil
shutil.copytree(str(BEST_CHECKPOINT), str(FINAL_ADAPTER_PATH), dirs_exist_ok=True)
print(f"Best checkpoint (epoch 10) copied to {FINAL_ADAPTER_PATH}")

Best checkpoint (epoch 10) copied to /content/drive/MyDrive/finetuning_checkpoints/final_adapter_epoch10


In [7]:
# save the final LoRA adapter (small — just the adapter weights,
# not the full model)
FINAL_ADAPTER_PATH = CHECKPOINT_DIR / "final_adapter"
model.save_pretrained(str(FINAL_ADAPTER_PATH))
tokenizer.save_pretrained(str(FINAL_ADAPTER_PATH))

print(f"Adapter saved to {FINAL_ADAPTER_PATH}")

NameError: name 'model' is not defined

In [15]:
# quick smoke test: does the fine-tuned model produce a
# reasonable, cited answer on a held-out-style prompt before you move to
# full evaluation?
test_messages = [
    {"role": "system", "content": CITATION_SYSTEM_PROMPT},
    {"role": "user", "content": "Context:\n[1] (source: test.pdf)\nPython was created by Guido van Rossum and released in 1991.\n\nQuestion: Who created Python and when?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.2,
    do_sample=True,
)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Python was created by Guido van Rossum in 1991. [1]
